## Config

In [1]:
import os, torch

CFG = {
    # Model
    "scale"       : 2,
    "dim"         : 48,
    "n_blocks"    : 6,
    "in_channels" : 3,
    "out_channels": 3,
    "device"      : "cuda",

    # Checkpoint cần load
    "checkpoint"  : "experiments/span_df2k_optuna_x2_Y/checkpoints/best.pth",

    # Các tập validate
    "val_sets" : {
        "DIV2K"   : ("Dataset/test/DIV2K_valid_HR",  "Dataset/test/DIV2K_valid_LR_bicubic/X2"),
        "Set5"    : ("Dataset/test/Set5_HR/X2",          "Dataset/test/Set5_LR/X2"),
        "Set14"   : ("Dataset/test/Set14_HR/X2",         "Dataset/test/Set14_LR/X2"),
        "BSD100"  : ("Dataset/test/BSD100_HR/X2",        "Dataset/test/BSD100_LR/X2"),
        "Urban100": ("Dataset/test/Urban100_HR/X2",      "Dataset/test/Urban100_LR/X2"),
    },
}

# Kiem tra checkpoint
ckpt_ok = os.path.isfile(CFG["checkpoint"])
print(f"Checkpoint : {'OK' if ckpt_ok else 'Không tìm thấy checkpoint'} — {CFG['checkpoint']}")

# Kiem tra val sets
print()
for name, (hr, lr) in CFG["val_sets"].items():
    hr_ok = os.path.isdir(hr)
    lr_ok = os.path.isdir(lr)
    status = "OK" if hr_ok and lr_ok else "Không tìm thấy"
    print(f"  {status:<15} {name:<10} HR={hr}  LR={lr}")


Checkpoint : OK — experiments/span_df2k_optuna_x2_Y/checkpoints/best.pth

  OK              DIV2K      HR=Dataset/test/DIV2K_valid_HR  LR=Dataset/test/DIV2K_valid_LR_bicubic/X2
  OK              Set5       HR=Dataset/test/Set5_HR/X2  LR=Dataset/test/Set5_LR/X2
  OK              Set14      HR=Dataset/test/Set14_HR/X2  LR=Dataset/test/Set14_LR/X2
  OK              BSD100     HR=Dataset/test/BSD100_HR/X2  LR=Dataset/test/BSD100_LR/X2
  OK              Urban100   HR=Dataset/test/Urban100_HR/X2  LR=Dataset/test/Urban100_LR/X2


## Kiến trúc SPAN

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SPAB(nn.Module):
    def __init__(self, dim=48):
        super().__init__()
        self.c1_r = nn.Conv2d(dim, dim, 3, 1, 1)
        self.c2_r = nn.Conv2d(dim, dim, 3, 1, 1)
        self.c3_r = nn.Conv2d(dim, dim, 3, 1, 1)
        self.act  = nn.LeakyReLU(0.2, inplace=True)

        self.attn_act = lambda x: torch.sigmoid(x) - 0.5

    def forward(self, x):
        # Feature extraction qua 3 conv layers
        h = self.act(self.c1_r(x))
        h = self.act(self.c2_r(h))
        h = self.c3_r(h)            # Hi — chưa activation


        u = x + h                   # Ui = Oi-1 ⊕ Hi

        # Attention map: symmetric activation (parameter-free)
        v = self.attn_act(h)             # Vi = σa(Hi)

        # Output: feature × attention map
        return u * v                # Oi = Ui ⊙ Vi



# Upsampler
class Upsampler(nn.Sequential):
    def __init__(self, scale, out_ch=3, dim=48):
        L = [
            nn.Conv2d(dim, out_ch * scale * scale, 3, 1, 1),
            nn.PixelShuffle(scale),
        ]
        super().__init__(*L)


# SPAN 
class SPAN(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, dim=48, n_blocks=6,
                 scale=4):
        super().__init__()
        self.scale    = scale
        self.n_blocks = n_blocks

        # Input conv
        self.conv_in = nn.Conv2d(in_ch, dim, 3, 1, 1)

        # n SPAB blocks
        self.blocks = nn.ModuleList([SPAB(dim) for _ in range(n_blocks)])


        # giảm về dim bằng 1×1 conv
        self.conv_cat = nn.Conv2d(dim * 4, dim, 1, 1, 0)

        # Upsampler + output conv
        self.up = Upsampler(scale, out_ch=out_ch, dim=dim)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                if m.weight.shape[1] not in (1, 4):  # bỏ qua depthwise và sobel
                    nn.init.trunc_normal_(m.weight, std=0.02)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)

    def forward(self, x):
        f = self.conv_in(x)   # input feature

        # Chạy qua tất cả SPAB blocks, lưu output của từng block
        outputs = []
        for block in self.blocks:
            f = block(f)
            outputs.append(f)
        cat = torch.cat([
            outputs[0],
            outputs[1],
            outputs[-2],
            outputs[-1],
        ], dim=1)               # (B, 4×dim, H, W)

        f = self.conv_cat(cat)  # (B, dim, H, W)

        return self.up(f)

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)
print("SPAN defined")

SPAN defined


## Metrics

In [3]:
import math
import torch
import torch.nn.functional as F


def calc_psnr(pred, target, max_val=1.0):
    with torch.no_grad():
        mse = F.mse_loss(pred.clamp(0, max_val), target.clamp(0, max_val))
        return float("inf") if mse == 0 else 10 * math.log10(max_val**2 / mse.item())


def calc_ssim(pred, target, win=11):
    with torch.no_grad():
        C1, C2 = 1e-4, 9e-4
        c = pred.shape[1]
        coords = torch.arange(win, dtype=torch.float32, device=pred.device) - win // 2
        g = torch.exp(-coords**2 / (2 * 1.5**2)); g /= g.sum()
        k = g.outer(g).unsqueeze(0).unsqueeze(0).expand(c, 1, win, win)
        p, t = pred.clamp(0, 1), target.clamp(0, 1)
        pad  = win // 2
        mu1 = F.conv2d(p, k, padding=pad, groups=c)
        mu2 = F.conv2d(t, k, padding=pad, groups=c)
        s1  = F.conv2d(p*p, k, padding=pad, groups=c) - mu1**2
        s2  = F.conv2d(t*t, k, padding=pad, groups=c) - mu2**2
        s12 = F.conv2d(p*t, k, padding=pad, groups=c) - mu1 * mu2
        return ((2*mu1*mu2+C1)*(2*s12+C2)/((mu1**2+mu2**2+C1)*(s1+s2+C2))).mean().item()


class MetricTracker:
    def __init__(self): self.reset()
    def reset(self): self._p = self._s = self._n = 0.0
    def update(self, pred, target):
        self._p += calc_psnr(pred, target)
        self._s += calc_ssim(pred, target)
        self._n += 1
    @property
    def avg_psnr(self): return self._p / max(self._n, 1)
    @property
    def avg_ssim(self): return self._s / max(self._n, 1)




def rgb_to_y(img):
    """Chuyển RGB tensor [B,3,H,W] → Y channel [B,1,H,W] (chuẩn BT.601)."""
    r, g, b = img[:, 0:1], img[:, 1:2], img[:, 2:3]
    return 0.257 * r + 0.504 * g + 0.098 * b + 16/255

def calc_psnr_y(pred, target):
    """PSNR trên Y channel — chuẩn so sánh với paper SR."""
    return calc_psnr(rgb_to_y(pred), rgb_to_y(target))

def calc_ssim_y(pred, target):
    """SSIM trên Y channel — chuẩn so sánh với paper SR."""
    return calc_ssim(rgb_to_y(pred), rgb_to_y(target))


class MetricTrackerY:
    """Track cả RGB lẫn Y-channel metrics."""
    def __init__(self): self.reset()
    def reset(self): self._p = self._s = self._py = self._sy = self._n = 0.0
    def update(self, pred, target):
        self._p  += calc_psnr(pred, target)
        self._s  += calc_ssim(pred, target)
        self._py += calc_psnr_y(pred, target)
        self._sy += calc_ssim_y(pred, target)
        self._n  += 1
    @property
    def avg_psnr(self):   return self._p  / max(self._n, 1)
    @property
    def avg_ssim(self):   return self._s  / max(self._n, 1)
    @property
    def avg_psnr_y(self): return self._py / max(self._n, 1)
    @property
    def avg_ssim_y(self): return self._sy / max(self._n, 1)


print("OK Metrics: PSNR, SSIM, PSNR-Y, SSIM-Y")



OK Metrics: PSNR, SSIM, PSNR-Y, SSIM-Y


## Load Model từ Checkpoint

In [4]:
import torch
from pathlib import Path

device = torch.device(CFG["device"] if torch.cuda.is_available() else "cpu")

model = SPAN(
    in_ch    = CFG["in_channels"],
    out_ch   = CFG["out_channels"],
    dim      = CFG["dim"],
    n_blocks = CFG["n_blocks"],
    scale    = CFG["scale"],
).to(device)

ckpt = torch.load(CFG["checkpoint"], map_location=device)

# Tương thích cả 2 kiểu lưu: dict có "model" key hoặc state_dict thẳng
state = ckpt.get("model", ckpt) if isinstance(ckpt, dict) else ckpt
model.load_state_dict(state)
model.eval()

epoch    = ckpt.get("epoch",     "?") if isinstance(ckpt, dict) else "?"
best_val = ckpt.get("best_psnr", "?") if isinstance(ckpt, dict) else "?"

print(f"OK Model loaded")
print(f"  Epoch     : {epoch}")
print(f"  Best PSNR : {best_val}")
print(f"  Device    : {device}")


OK Model loaded
  Epoch     : 290
  Best PSNR : 35.4324035481004
  Device    : cuda


## Validate trên tất cả dataset

In [5]:
import os, time
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from tqdm.auto import tqdm

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def _get_paths(folder):
    paths = sorted([os.path.join(folder, f) for f in os.listdir(folder)
                    if Path(f).suffix.lower() in IMG_EXTS])
    assert paths, f"Khong tim thay anh trong: {folder}"
    return paths


class ValidDataset(Dataset):
    def __init__(self, hr_dir, lr_dir):
        self.hr, self.lr = _get_paths(hr_dir), _get_paths(lr_dir)
        assert len(self.hr) == len(self.lr), (
            f"So anh HR ({len(self.hr)}) != LR ({len(self.lr)})")
    def __len__(self): return len(self.hr)
    def __getitem__(self, i):
        return (TF.to_tensor(Image.open(self.lr[i]).convert("RGB")),
                TF.to_tensor(Image.open(self.hr[i]).convert("RGB")))


def measure_inference_ms(model, loader, device, warmup=0):
    """Đo thời gian inference trung bình mỗi ảnh (ms).

    - Warmup: chạy vài ảnh đầu để GPU ổn định, không tính vào kết quả
    - Dùng torch.cuda.Event để đo chính xác trên GPU
    - Dùng time.perf_counter nếu chạy trên CPU
    """
    use_cuda = device.type == "cuda"
    times = []

    with torch.no_grad():
        for i, (lr_img, _) in enumerate(loader):
            lr_img = lr_img.to(device)

            if use_cuda:
                start = torch.cuda.Event(enable_timing=True)
                end   = torch.cuda.Event(enable_timing=True)
                torch.cuda.synchronize()
                start.record()
                _ = model(lr_img)
                end.record()
                torch.cuda.synchronize()
                elapsed = start.elapsed_time(end)   # ms
            else:
                t0 = time.perf_counter()
                _  = model(lr_img)
                elapsed = (time.perf_counter() - t0) * 1000  # ms

            if i >= warmup:   # bỏ qua warmup frames
                times.append(elapsed)

    if not times:
        return 0.0, 0.0
    avg = sum(times) / len(times)
    mn  = min(times)
    return avg, mn


# ── Chạy validate ─────────────────────────────────────────────────────────────
results = {}

header = (f"{'Dataset':<10} | {'Ảnh':>5} | {'PSNR-RGB':>9} | {'SSIM-RGB':>9}"
          f" | {'PSNR-Y':>8} | {'SSIM-Y':>8} | {'ms/img':>8}")
print(header)
print("─" * len(header))

for name, (hr_path, lr_path) in CFG["val_sets"].items():
    if not (os.path.isdir(hr_path) and os.path.isdir(lr_path)):
        print(f"{name:<10} | SKIP — thu muc khong ton tai")
        continue

    ds      = ValidDataset(hr_path, lr_path)
    loader  = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)
    tracker = MetricTrackerY()

    # ── Đo inference time ─────────────────────────────────────────────────
    avg_ms, min_ms = measure_inference_ms(model, loader, device, warmup=0)

    # ── Tính metrics ──────────────────────────────────────────────────────
    with torch.no_grad():
        for lr_img, hr_img in tqdm(loader, desc=name, leave=False):
            sr = model(lr_img.to(device))
            tracker.update(sr, hr_img.to(device))

    results[name] = {
        "psnr"  : tracker.avg_psnr,
        "ssim"  : tracker.avg_ssim,
        "psnr_y": tracker.avg_psnr_y,
        "ssim_y": tracker.avg_ssim_y,
        "ms_avg": avg_ms,
        "ms_min": min_ms,
        "n"     : len(ds),
    }
    print(f"{name:<10} | {len(ds):>5}"
          f" | {tracker.avg_psnr:>9.4f} | {tracker.avg_ssim:>9.4f}"
          f" | {tracker.avg_psnr_y:>8.4f} | {tracker.avg_ssim_y:>8.4f}"
          f" | {avg_ms:>7.2f}ms")

print("─" * len(header))


Dataset    |   Ảnh |  PSNR-RGB |  SSIM-RGB |   PSNR-Y |   SSIM-Y |   ms/img
───────────────────────────────────────────────────────────────────────────


DIV2K:   0%|          | 0/100 [00:00<?, ?it/s]

DIV2K      |   100 |   33.9519 |    0.9346 |  35.4324 |   0.9427 |  179.76ms


Set5:   0%|          | 0/5 [00:00<?, ?it/s]

Set5       |     5 |   35.2893 |    0.9450 |  37.4239 |   0.9614 |    8.03ms


...

Set14      |    14 |   30.7660 |    0.8938 |  32.7923 |   0.9176 |   12.52ms


BSD100:   0%|          | 0/100 [00:00<?, ?it/s]

BSD100     |   100 |   30.4398 |    0.8940 |  31.7870 |   0.9029 |    6.78ms


Urban100:   0%|          | 0/100 [00:00<?, ?it/s]

Urban100   |   100 |   29.0741 |    0.9072 |  30.6390 |   0.9171 |   47.37ms
───────────────────────────────────────────────────────────────────────────
